In [1]:
# =============================================================
# 04_company_financials.ipynb
# Semiconductor Supply Chain Analysis
# Author: Ray Garza | Austin TX | March 2026
# Purpose: Pull annual financials for key semiconductor companies
#          via yfinance (sourced from SEC filings via Yahoo Finance)
#          Output: clean DataFrame ready for Snowflake load
# =============================================================

import yfinance as yf
import pandas as pd

# Confirm versions
print(f"yfinance version: {yf.__version__}")
print(f"pandas version: {pd.__version__}")
print("Notebook initialized.")

yfinance version: 1.2.0
pandas version: 2.3.3
Notebook initialized.


In [2]:
# Define the companies we're analyzing
# Three layers of the semiconductor stack represented:
# Equipment makers, foundries, fabless designers, memory, sub-tier

COMPANIES = {
    "AMAT":     {"name": "Applied Materials",  "type": "equipment"},
    "ASML":     {"name": "ASML",               "type": "equipment"},
    "LRCX":     {"name": "Lam Research",       "type": "equipment"},
    "KLAC":     {"name": "KLA Corporation",    "type": "equipment"},
    "UCTT":     {"name": "Ultra Clean Holdings","type": "sub_tier"},
    "TSM":      {"name": "TSMC",               "type": "foundry"},
    "INTC":     {"name": "Intel",              "type": "foundry"},
    "005930.KS":{"name": "Samsung",            "type": "foundry"},
    "MU":       {"name": "Micron",             "type": "memory"},
    "NVDA":     {"name": "NVIDIA",             "type": "fabless"},
}

print(f"Companies to pull: {len(COMPANIES)}")
for ticker, meta in COMPANIES.items():
    print(f"  {ticker:<12} {meta['name']:<25} [{meta['type']}]")

Companies to pull: 10
  AMAT         Applied Materials         [equipment]
  ASML         ASML                      [equipment]
  LRCX         Lam Research              [equipment]
  KLAC         KLA Corporation           [equipment]
  UCTT         Ultra Clean Holdings      [sub_tier]
  TSM          TSMC                      [foundry]
  INTC         Intel                     [foundry]
  005930.KS    Samsung                   [foundry]
  MU           Micron                    [memory]
  NVDA         NVIDIA                    [fabless]


In [3]:
# Pull annual financials from Yahoo Finance (sourced from SEC 10-K filings)
# Currency conversions applied at source — all output in USD billions:
#   Samsung (005930.KS): KRW → USD at 1,300 KRW/USD (Fed H.10 avg 2022-2025)
#   TSMC (TSM): NTD → USD at 31 NTD/USD (Fed H.10 avg 2022-2025)
#   All others: already USD, divide by 1e9 for billions
# Capex flipped to positive — cash flow statements report it as negative outflow
# Verified: TSMC 2024 ~$93B, AMAT 2024 ~$27B match published annual reports

records = []

for ticker, meta in COMPANIES.items():
    print(f"Pulling {meta['name']} ({ticker})...")
    
    try:
        stock = yf.Ticker(ticker)
        income = stock.financials
        cashflow = stock.cashflow
        years = income.columns
        
        for year in years:
            try:
                revenue = income.loc["Total Revenue", year] if "Total Revenue" in income.index else None
                rd = income.loc["Research And Development", year] if "Research And Development" in income.index else None
                capex_raw = cashflow.loc["Capital Expenditure", year] if "Capital Expenditure" in cashflow.index else None
                capex = abs(capex_raw) if capex_raw is not None else None

                # Convert to USD at source before storing
                if ticker == "005930.KS":
                    divisor = 1e9 * 1300  # KRW billions to USD billions
                elif ticker == "TSM":
                    divisor = 1e9 * 31    # NTD to USD billions
                else:
                    divisor = 1e9         # Already USD, just convert to billions

                records.append({
                    "ticker": ticker,
                    "company_name": meta["name"],
                    "company_type": meta["type"],
                    "fiscal_year": year.year,
                    "revenue_b": round(revenue / divisor, 3) if revenue else None,
                    "rd_spend_b": round(rd / divisor, 3) if rd else None,
                    "capex_b": round(capex / divisor, 3) if capex else None,
                    "currency_note": (
                        "USD_billions_converted_from_KRW_at_1300" if ticker == "005930.KS"
                        else "USD_billions_converted_from_NTD_at_31" if ticker == "TSM"
                        else "USD_billions"
                    )
                })
            except Exception as e:
                print(f"  ⚠ Year {year.year} error: {e}")
                
    except Exception as e:
        print(f"  ✗ Failed {ticker}: {e}")

print(f"\nTotal records: {len(records)}")

Pulling Applied Materials (AMAT)...
Pulling ASML (ASML)...
Pulling Lam Research (LRCX)...
Pulling KLA Corporation (KLAC)...
Pulling Ultra Clean Holdings (UCTT)...
Pulling TSMC (TSM)...
Pulling Intel (INTC)...
Pulling Samsung (005930.KS)...
Pulling Micron (MU)...
Pulling NVIDIA (NVDA)...

Total records: 47


In [4]:
# Build DataFrame.. currency already converted in Cell 3, do not re-convert
# All monetary values in USD billions at this point

# Convert to DataFrame and inspect
df_financials = pd.DataFrame(records)

# Drop rows where all financial columns are NaN
df_financials = df_financials.dropna(
    subset=["revenue_b", "rd_spend_b", "capex_b"],
    how="all"
).reset_index(drop=True)

df_financials = df_financials.sort_values(
    ["ticker", "fiscal_year"]
).reset_index(drop=True)

print(f"Shape: {df_financials.shape}")
print(f"\nTSMC:")
print(df_financials[df_financials["ticker"] == "TSM"][
    ["fiscal_year", "revenue_b", "capex_b"]
])
print(f"\nSamsung:")
print(df_financials[df_financials["ticker"] == "005930.KS"][
    ["fiscal_year", "revenue_b", "capex_b"]
])

Shape: (40, 8)

TSMC:
    fiscal_year  revenue_b  capex_b
32         2022     73.029   35.149
33         2023     69.733   30.819
34         2024     93.365   31.128
35         2025    122.873   41.374

Samsung:
   fiscal_year  revenue_b  capex_b
0         2022    232.486   40.867
1         2023    199.181   46.565
2         2024    231.439   41.340
3         2025    256.620   40.118


In [5]:
# Sanity check before saving
# Expected 2024 revenue (USD B): TSMC ~$93, Samsung ~$231, AMAT ~$27, NVDA ~$61, UCTT ~$2

negative_revenue = df_financials[df_financials["revenue_b"] < 0]
print(f"Negative revenue rows: {len(negative_revenue)}")

print(f"\nAverage 2024 revenue by company type (USD billions):")
summary = (df_financials[df_financials["fiscal_year"] == 2024]
    .groupby("company_type")[["revenue_b", "capex_b"]]
    .mean()
    .round(3)
)
print(summary)

print(f"\nFinal shape: {df_financials.shape}")
print(f"Years: {sorted(df_financials['fiscal_year'].unique())}")
print(f"Companies: {df_financials['ticker'].nunique()}")

Negative revenue rows: 0

Average 2024 revenue by company type (USD billions):
              revenue_b  capex_b
company_type                    
equipment        20.039    0.987
fabless          60.922    1.069
foundry         125.968   32.137
memory           25.111    8.386
sub_tier          2.098    0.064

Final shape: (40, 8)
Years: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Companies: 10


In [7]:
import os

# Save locally as CSV — bridge until Databricks/Snowflake pipeline is complete
# Final destination: Snowflake SEMICON_DB.GOLD.COMPANY_FINANCIALS
# Also write as Parquet to Databricks silver layer
# to /Volumes/main/semiconductors/silver_financials/

output_path = "../raw_datasets/company_financials.csv"
df_financials.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Size: {os.path.getsize(output_path) / 1024:.1f} KB")
df_financials.tail(10)

Saved: ../raw_datasets/company_financials.csv
Size: 2.7 KB


,ticker,company_name,company_type,fiscal_year,revenue_b,rd_spend_b,capex_b,currency_note
30,NVDA,NVIDIA,fabless,2025,130.497,12.914,3.236,USD_billions
31,NVDA,NVIDIA,fabless,2026,215.938,18.497,6.042,USD_billions
32,TSM,TSMC,foundry,2022,73.029,5.267,35.149,USD_billions_converted_from_NTD_at_31
33,TSM,TSMC,foundry,2023,69.733,5.883,30.819,USD_billions_converted_from_NTD_at_31
34,TSM,TSMC,foundry,2024,93.365,6.587,31.128,USD_billions_converted_from_NTD_at_31
35,TSM,TSMC,foundry,2025,122.873,7.949,41.374,USD_billions_converted_from_NTD_at_31
36,UCTT,Ultra Clean Holdings,sub_tier,2022,2.374,0.028,0.100,USD_billions
37,UCTT,Ultra Clean Holdings,sub_tier,2023,1.734,0.028,0.076,USD_billions
38,UCTT,Ultra Clean Holdings,sub_tier,2024,2.098,0.028,0.064,USD_billions
39,UCTT,Ultra Clean Holdings,sub_tier,2025,2.054,0.032,0.050,USD_billions
